# Kuon LLM — Colab GPU quickstart

Latih `kuon-13m` dari nol sampai bisa menulis cerita, di GPU gratis Colab.

**Cara pakai:**
1. Upload `kuon-llm.zip` (ada di folder `notebooks/` repo) lewat ikon folder 📁 di panel kiri Colab, atau lewat dialog upload di bawah.
2. Jalankan sel 1 → 4 berurutan (Shift+Enter).
3. Notebook ini sudah menyertakan tokenizer Kuon (8192 vocab) di dalam zip, jadi checkpoint-nya kompatibel dengan training lokal di laptop.

⚠️ Runtime → Change runtime type → **T4 GPU** dulu.

## 1. Setup (~1 menit)
Upload `kuon-llm.zip` saat diminta (atau lewat panel Files kiri), lalu jalankan sel ini.

In [ ]:
#@title 1. Setup Kuon
import pathlib, subprocess, sys
if not pathlib.Path("kuon-llm.zip").exists():
    from google.colab import files
    files.upload()  # pilih kuon-llm.zip
!unzip -oq kuon-llm.zip -d /content/repo
%cd /content/repo
!pip install -q -e .
import torch
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — ganti runtime ke T4!")

## 2. Data (~20-30 menit)
Download TinyStories, pakai tokenizer yang sudah terlatih, lalu pack 200.000 cerita (= ±44 juta token) ke `train.bin`/`val.bin`.

In [ ]:
#@title 2. Siapkan dataset
!python scripts/prepare_data.py --source tinystories --out data/tinystories --workers 2 --limit 200000

## 3. Training (~25-35 menit di T4)
2.000 step × 16.384 token = 32,8 juta token. Loss harus turun dari ±9.0 menuju 1.x; sampel cerita muncul tiap 100 step.

In [ ]:
#@title 3. Latih kuon-13m
!python scripts/train.py --config configs/kuon-13m.json --device cuda

## 4. Coba model (~10 detik)
Prompt bahasa Inggris — model ini base model: ia MELANJUTKAN cerita, bukan menjawab chat (itu tugas tahap SFT).

In [ ]:
#@title 4. Generate
!python scripts/generate.py --checkpoint checkpoints/kuon-13m/kuon-13m-last.pt --prompt "Once upon a time" --tokens 150

## Opsional: simpan checkpoint ke Google Drive
Colab gratis menghapus VM saat sesi berakhir. Mount Drive agar checkpoint bisa dipakai lanjut (resume) di sesi berikutnya:

```python
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kuon && cp -r checkpoints /content/drive/MyDrive/kuon/
```

Lalu sesi berikutnya: `!python scripts/train.py --config configs/kuon-13m.json --device cuda --resume /content/drive/MyDrive/kuon/checkpoints/kuon-13m/kuon-13m-last.pt`

## Naik skala
- `kuon-48m`: ganti `--config configs/kuon-48m.json`, data penuh (`--limit 0` → 2,1 juta cerita), dan `--resume` dari checkpoint Drive.
- Kaggle (30 jam GPU/minggu) lebih longgar dari Colab untuk run panjang.